# Phase 4 — RAGAS 평가 (`02_ragas_eval.ipynb`)

Phase 3에서 구성한 RAG 파이프라인을 **골든셋 50개**로 실행하고, **Claude(Haiku 4.5)** 를 평가자(judge)로 사용해 4개 지표를 측정한다.

- `faithfulness` — 답변이 검색된 문맥에 충실한가 (환각 여부)
- `answer_relevancy` — 답변이 질문에 적절한가
- `context_precision` — 검색된 문맥 중 정답에 기여하는 비율 (정답 기준)
- `context_recall` — 정답을 뒷받침하는 문맥이 얼마나 검색되었는가 (정답 기준)

### 실행 전 준비 (필수)
1. **Phase 3 완료** — `01_pipeline.ipynb`로 pgvector 테이블 `rag_docs_bge_m3_ko`가 적재된 상태여야 한다.
2. **Ollama 구동** — `gemma4:e2b` 모델 로드 (`ollama serve`).
3. **`.env`에 `ANTHROPIC_API_KEY`** 설정.

### 순차 실행 시 알아둘 점
- **셀 순서대로** 위에서 아래로 실행한다.
- **무거운 단계는 디스크 캐싱**된다 — 골든셋 RAG 실행 결과는 `results/eval_inputs.json`에 저장되어, RAGAS 코드를 다시 돌릴 때 Ollama 추론 50회를 반복하지 않는다. (`FORCE_RECOLLECT=True`로 강제 재수집)
- RAGAS `evaluate()`는 골든셋×지표만큼 **Claude API를 호출**하므로 시간·비용이 든다.

## 0. 호환성 셰임 (RAGAS 0.4.3 × langchain 1.x)

> ⚠️ **이 셀을 `ragas` import보다 먼저 실행해야 한다.**

RAGAS 0.4.3(최신)은 `llms/base.py`에서 `langchain_community.chat_models.vertexai.ChatVertexAI`를 **모듈 로드 시 무조건 import**한다.
그런데 이 프로젝트가 쓰는 langchain 1.x 라인의 `langchain-community 0.4.x`는 sunset되며 해당 경로를 삭제했다 → `ModuleNotFoundError`.

우리는 Anthropic 래퍼만 쓰므로 Vertex 경로는 실제로 필요 없다. 아래처럼 **빈 자리표시 모듈을 주입**해 import만 통과시키면 된다.
(전체 스택을 0.3.x로 다운그레이드하면 Phase 3의 langchain 1.x API가 깨지므로, 격리된 셰임이 더 안전하다. RAGAS가 langchain 1.x를 지원하면 이 셀은 삭제 가능.)

In [1]:
import sys
import types

# langchain 1.x에서 삭제된 langchain_community.chat_models.vertexai 경로를 자리표시로 복원
import langchain_community.chat_models as _cm

if 'langchain_community.chat_models.vertexai' not in sys.modules:
    _vmod = types.ModuleType('langchain_community.chat_models.vertexai')

    class ChatVertexAI:  # 자리표시 — Vertex를 선택하지 않는 한 사용되지 않음
        ...

    _vmod.ChatVertexAI = ChatVertexAI
    sys.modules['langchain_community.chat_models.vertexai'] = _vmod
    _cm.vertexai = _vmod

import langchain_community.llms as _llms
if not hasattr(_llms, 'VertexAI'):
    class VertexAI:  # 자리표시
        ...
    _llms.VertexAI = VertexAI

print('호환성 셰임 적용 완료 — 이제 ragas를 import해도 안전합니다.')

/var/folders/fn/z8hqtckn0kzcm6gzx5dxdtx00000gn/T/ipykernel_57051/1599170419.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community.chat_models as _cm


호환성 셰임 적용 완료 — 이제 ragas를 import해도 안전합니다.


## 1. 설정 & 임포트

Phase 3와 **동일한** 임베딩 모델·컬렉션·접속정보를 재사용한다. 평가자 LLM만 새로 추가한다.

In [2]:
import json
import os
import warnings
from pathlib import Path

from dotenv import load_dotenv

load_dotenv('../.env')

# ragas.metrics는 v1.0에서 ragas.metrics.collections로 이전 예정 — 학습용으로 안정 API(evaluate)를 쓰므로
# 발생하는 DeprecationWarning은 의도된 것이라 숨긴다.
warnings.filterwarnings('ignore', category=DeprecationWarning)

# --- 경로 ---
GOLDEN_PATH = Path('../data/golden_set.json')
RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(exist_ok=True)
EVAL_INPUTS_PATH = RESULTS_DIR / 'eval_inputs.json'      # 골든셋 RAG 실행 결과 캐시
RAGAS_RESULT_PATH = RESULTS_DIR / 'ragas_baseline.json'  # 최종 점수
RAGAS_DETAIL_PATH = RESULTS_DIR / 'ragas_baseline_detail.csv'

# --- Phase 3와 동일한 임베딩/컬렉션 ---
EMBED_MODEL = 'dragonkue/BGE-m3-ko'
COLLECTION_NAME = 'rag_docs_bge_m3_ko'
TOP_K = 4

# --- 평가자(judge) LLM: Claude Haiku 4.5 (저렴·빠름, 평가용 적합) ---
JUDGE_MODEL = 'claude-haiku-4-5'

# --- pgvector 접속 (Phase 3와 동일) ---
PG_CONN = (
    f"postgresql+psycopg://{os.getenv('PGVECTOR_USER', 'user')}:"
    f"{os.getenv('PGVECTOR_PASSWORD', 'test123!')}@"
    f"{os.getenv('PGVECTOR_HOST', 'localhost')}:"
    f"{os.getenv('PGVECTOR_PORT', '5432')}/"
    f"{os.getenv('PGVECTOR_DB', 'ragdb')}"
)

golden_set = json.loads(GOLDEN_PATH.read_text(encoding='utf-8'))
assert os.getenv('ANTHROPIC_API_KEY'), 'ANTHROPIC_API_KEY가 .env에 없습니다.'
print(f'골든셋: {len(golden_set)}개 / 평가자: {JUDGE_MODEL}')

골든셋: 50개 / 평가자: claude-haiku-4-5


## 2. RAG 파이프라인 재구성 (Phase 3 재사용)

Phase 3에서 **이미 적재된** pgvector 테이블에 연결한다. 여기서는 재인덱싱하지 않는다 —
`init_vectorstore_table(overwrite_existing=True)`를 호출하면 데이터가 지워지므로 **호출하지 않는다.**

In [3]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={'device': device},
    encode_kwargs={'normalize_embeddings': True, 'batch_size': 32},
)
print(f'임베딩 로드 완료 (device={device}, dim={len(embeddings.embed_query("테스트"))})')

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

임베딩 로드 완료 (device=mps, dim=1024)


In [4]:
from langchain_postgres import PGEngine, PGVectorStore

# 기존 테이블에 연결만 한다 (재생성 X)
engine = PGEngine.from_connection_string(url=PG_CONN)
vectorstore = PGVectorStore.create_sync(
    engine=engine,
    table_name=COLLECTION_NAME,
    embedding_service=embeddings,
)
retriever = vectorstore.as_retriever(search_kwargs={'k': TOP_K})

# 적재 여부 확인 — 비어 있으면 Phase 3를 먼저 실행해야 한다
_probe = vectorstore.similarity_search(golden_set[0]['question'], k=1)
assert _probe, 'pgvector 테이블이 비어 있습니다. 먼저 01_pipeline.ipynb로 적재하세요.'
print(f'pgvector 연결 OK — 샘플 검색 결과 {len(_probe)}건')

pgvector 연결 OK — 샘플 검색 결과 1건


In [5]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_ollama import ChatOllama

rag_llm = ChatOllama(
    model=os.getenv('OLLAMA_MODEL', 'gemma4:e2b'),
    base_url=os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434'),
    temperature=0,
)

prompt = ChatPromptTemplate.from_template(
    '당신은 주어진 문맥(context)에만 근거해 한국어로 간결히 답하는 어시스턴트입니다.\n'
    '문맥에서 답을 찾을 수 없으면 "문맥에서 답을 찾을 수 없습니다."라고 답하세요.\n\n'
    '문맥:\n{context}\n\n'
    '질문: {question}\n'
    '답변:'
)

def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

answer_chain = prompt | rag_llm | StrOutputParser()

# 질문 → (검색 문서 + 생성 답변) 동시 반환 (Phase 3와 동일한 LCEL 구성)
qa_chain = RunnableParallel(
    question=RunnablePassthrough(),
    source_documents=retriever,
).assign(
    answer=lambda x: answer_chain.invoke(
        {'context': format_docs(x['source_documents']), 'question': x['question']}
    )
)
print('RAG 체인 구성 완료 — Ollama 연결 확인:', rag_llm.invoke('안녕').content[:40])

RAG 체인 구성 완료 — Ollama 연결 확인: 안녕하세요! 😊


## 3. 골든셋으로 RAG 실행 → 평가 입력 수집 (⏳ 무거움, 캐싱됨)

골든셋 50개 질문을 RAG 체인에 통과시켜 `(user_input, retrieved_contexts, response, reference)`를 모은다.
Ollama 추론 50회라 수 분 걸릴 수 있어 결과를 `results/eval_inputs.json`에 캐싱한다.

- 다시 수집하려면 `FORCE_RECOLLECT = True`로 바꾼다.
- RAGAS의 `SingleTurnSample` 필드명에 맞춰 키를 구성한다:
  `user_input`(질문) / `retrieved_contexts`(검색 청크들) / `response`(RAG 답변) / `reference`(정답).

In [6]:
FORCE_RECOLLECT = False

if EVAL_INPUTS_PATH.exists() and not FORCE_RECOLLECT:
    records = json.loads(EVAL_INPUTS_PATH.read_text(encoding='utf-8'))
    print(f'캐시 로드: {len(records)}개 ({EVAL_INPUTS_PATH})')
else:
    records = []
    n = len(golden_set)
    for i, item in enumerate(golden_set, 1):
        q = item['question']
        try:
            out = qa_chain.invoke(q)
            records.append({
                'user_input': q,
                'retrieved_contexts': [d.page_content for d in out['source_documents']],
                'response': out['answer'].strip(),
                'reference': item['ground_truth'],
            })
        except Exception as e:
            print(f'  [{i}/{n}] 실패: {e!r} — 건너뜀')
        if i % 5 == 0 or i == n:
            print(f'  진행 {i}/{n}')
    EVAL_INPUTS_PATH.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding='utf-8')
    print(f'수집 완료 & 저장: {len(records)}개 → {EVAL_INPUTS_PATH}')

# 첫 샘플 미리보기
_s = records[0]
print('\n[샘플 0]')
print('  질문 :', _s['user_input'])
print('  답변 :', _s['response'][:120])
print('  정답 :', _s['reference'])
print('  문맥 수:', len(_s['retrieved_contexts']))

  진행 5/50
  진행 10/50
  진행 15/50
  진행 20/50
  진행 25/50
  진행 30/50
  진행 35/50
  진행 40/50
  진행 45/50
  진행 50/50
수집 완료 & 저장: 50개 → ../results/eval_inputs.json

[샘플 0]
  질문 : 구식 군인들의 월급인 쌀에 모래와 돌멩이가 들어가있던 사건을 말미암아 일어난 사태의 이름은?
  답변 : 문맥에서 답을 찾을 수 없습니다.
  정답 : 임오군란
  문맥 수: 4


## 4. RAGAS 평가자(LLM·임베딩) 설정

- **평가자 LLM**: `ChatAnthropic`(Claude Haiku 4.5)을 `LangchainLLMWrapper`로 감싼다.
  Haiku 4.5는 `temperature`를 지원하므로 일관된 채점을 위해 `0`으로 둔다.
- **임베딩**: `answer_relevancy`가 임베딩 유사도를 쓰므로 동일한 BGE-m3-ko를 `LangchainEmbeddingsWrapper`로 감싼다.

In [7]:
from langchain_anthropic import ChatAnthropic
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

judge = ChatAnthropic(
    model=JUDGE_MODEL,
    temperature=0,
    max_tokens=1024,
    max_retries=3,
)
evaluator_llm = LangchainLLMWrapper(judge)
evaluator_emb = LangchainEmbeddingsWrapper(embeddings)
print('평가자 설정 완료:', JUDGE_MODEL)

평가자 설정 완료: claude-haiku-4-5


## 5. EvaluationDataset 구성

수집한 레코드를 `SingleTurnSample` 리스트로 만들어 `EvaluationDataset`에 담는다.

In [8]:
from ragas import EvaluationDataset
from ragas.dataset_schema import SingleTurnSample

samples = [
    SingleTurnSample(
        user_input=r['user_input'],
        retrieved_contexts=r['retrieved_contexts'],
        response=r['response'],
        reference=r['reference'],
    )
    for r in records
]
eval_dataset = EvaluationDataset(samples=samples)
print(f'EvaluationDataset 구성: {len(eval_dataset)}개 샘플')

EvaluationDataset 구성: 50개 샘플


## 6. 4개 지표 측정 (⏳ Claude API 호출)

| 지표 | 필요 입력 | 사용 자원 |
|---|---|---|
| `faithfulness` | response, retrieved_contexts | LLM |
| `answer_relevancy` | user_input, response | LLM + 임베딩 |
| `context_precision` | retrieved_contexts, reference | LLM |
| `context_recall` | retrieved_contexts, reference | LLM |

`evaluate()`에 `llm`·`embeddings`를 넘기면 각 지표에 자동 주입된다.

In [9]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
from ragas.run_config import RunConfig

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

# max_workers를 낮춰 API rate limit을 완화한다.
run_config = RunConfig(timeout=180, max_retries=5, max_workers=4)

result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=evaluator_llm,
    embeddings=evaluator_emb,
    run_config=run_config,
    show_progress=True,
)
print(result)

Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

Exception raised in Job[41]: RateLimitError(Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 50 requests per minute (org: 0cd2bc5e-3cec-49ad-9a2e-902b62618d1d, model: claude-haiku-4-5-20251001). For details, refer to: https://docs.claude.com/en/api/rate-limits. You can see the response headers for current usage. Try again later. View your current limits at https://console.anthropic.com/settings/limits. To raise this limit, purchase credits to advance to the next usage tier at https://console.anthropic.com/settings/billing."}, 'request_id': 'req_011CbotMrzyunh2JrTn2nzjf'})
Exception raised in Job[45]: RateLimitError(Error code: 429 - {'type': 'error', 'error': {'type': 'rate_limit_error', 'message': "This request would exceed your organization's rate limit of 50 requests per minute (org: 0cd2bc5e-3cec-49ad-9a2e-902b62618d1d, model: claude-haiku-4-5-20251001). For details, refer to: https://d

{'faithfulness': 0.3250, 'answer_relevancy': 0.0698, 'context_precision': 0.1700, 'context_recall': 0.1800}


## 7. 결과 저장 & 집계

- 샘플별 점수 → `results/ragas_baseline_detail.csv`
- 지표별 평균 → `results/ragas_baseline.json`

In [10]:
import pandas as pd  # noqa: F401  (to_pandas 내부 사용)

df = result.to_pandas()
df.to_csv(RAGAS_DETAIL_PATH, index=False, encoding='utf-8-sig')

metric_cols = [c for c in df.columns if c in ('faithfulness', 'answer_relevancy', 'context_precision', 'context_recall')]
aggregate = {c: float(df[c].mean(skipna=True)) for c in metric_cols}
RAGAS_RESULT_PATH.write_text(json.dumps(aggregate, ensure_ascii=False, indent=2), encoding='utf-8')

print('=== 지표별 평균 점수 ===')
for k, v in aggregate.items():
    print(f'  {k:20s}: {v:.4f}')
print(f'\n저장: {RAGAS_RESULT_PATH}\n      {RAGAS_DETAIL_PATH}')
df.head()

=== 지표별 평균 점수 ===
  faithfulness        : 0.3250
  answer_relevancy    : 0.0698
  context_precision   : 0.1700
  context_recall      : 0.1800

저장: ../results/ragas_baseline.json
      ../results/ragas_baseline_detail.csv


,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall
0,구식 군인들의 월급인 쌀에 모래와 돌멩이가 들어가있던 사건을 말미암아 일어난 사태의...,"[4월 27일에 여섯 번째로 기우제를 지냈다. 그러나 이틀 뒤인 4월 29일, 평안...",문맥에서 답을 찾을 수 없습니다.,임오군란,0.0,0.000000,0.0,0.0
1,백련교도의 난 때 수세에 몰린 왕총아가 전투 도중 절벽에서 뛰어내려 자진한 때는 언...,[. 수세에 몰린 왕총아는 1798년(가경 3년) 전투 도중 절벽에서 뛰어내려 자진...,1798년(가경 3년),1798년,1.0,0.233800,1.0,1.0
2,고려대학교 교호에 등장하는 몇 사람의 이름이 등장하는가?,"[이처럼 고려대학교 교호에는 입실렌티, 체이홉, 카시코시 코시코, 카를 마르크스까지...",네 사람의 이름이 등장한다.,네 사람,1.0,0.530109,1.0,1.0
3,주병진이 12년 만에 출연한 프로그램은?,[이후 공효진은 코믹한 연기에서 잠시 벗어나 2003년 처제와 형부의 사랑이야기라는...,문맥에서 답을 찾을 수 없습니다.,무릎팍도사,0.0,0.000000,0.0,0.0
4,서재필은 어떤 사람들에게 도움을 요청했는가?,"[이에 김옥균, 박영효, 서재필 등의 급진개화파들은 일본식 서구화를 부르짖었다가 민...",문맥에서 답을 찾을 수 없습니다.,한인대회에 참석한 [[미국인,0.0,0.000000,0.0,0.0


## 8. 지표 해석 메모

점수는 모두 **0~1**(높을수록 좋음).

- **faithfulness (충실도)** — 낮으면 RAG 답변이 문맥에 없는 내용을 지어냄(환각). 로컬 소형 LLM(gemma4:e2b)에서는 종종 낮게 나온다.
- **answer_relevancy (답변 적절성)** — 낮으면 질문과 동떨어진/장황한 답변. 프롬프트로 "간결히"를 지시했으므로 비교적 높게 나오는 편.
- **context_precision (문맥 정밀도)** — 검색된 top-k 중 정답에 기여한 청크 비율. 낮으면 리트리버가 노이즈를 많이 가져옴 → `TOP_K`나 임베딩 개선 여지.
- **context_recall (문맥 재현율)** — 정답을 뒷받침하는 내용이 검색에 얼마나 포함됐는가. **낮으면 리트리버가 정답 근거를 못 찾은 것** — Phase 5 벡터DB/임베딩 비교의 핵심 지표.

### ⚠️ 주의 — Phase 3의 `MAX_DOCS` 영향
Phase 3에서 `MAX_DOCS=200`으로 **일부 문서만 색인**했다면, 골든셋 질문의 정답 문서가 색인에 없을 수 있어 `context_recall`이 구조적으로 낮게 나온다.
제대로 된 baseline을 보려면 `01_pipeline.ipynb`에서 `MAX_DOCS=None`(전체 색인)로 다시 적재한 뒤, 이 노트북의 `FORCE_RECOLLECT=True`로 재수집해서 평가하라.

이 baseline 점수가 **Phase 5(벡터DB·임베딩 비교)** 의 기준선이 된다.